# Análise de Índices de Vegetação

Pipeline de visão computacional para cálculo de VARI e ExG a partir de imagens RGB.

## Etapa 1 — Carregar imagem e converter BGR → RGB

In [1]:
import cv2
import numpy as np

# cv2.imread retorna um array NumPy (H, W, 3) em BGR com dtype uint8
img_bgr = cv2.imread('../images/paisagem-verde.jpeg')

# OpenCV carrega em BGR; Matplotlib exibe em RGB — a conversão corrige a ordem dos canais
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

## Etapa 2 — Inspecionar shape, dtype e pixel

In [2]:
# shape retorna (altura, largura, canais) — eixo 0 é vertical, eixo 1 horizontal
print('Shape:', img_rgb.shape)

# dtype indica o tipo de cada elemento; uint8 = inteiro sem sinal de 0 a 255
print('Dtype:', img_rgb.dtype)

# img[y, x] acessa o pixel na linha y, coluna x — retorna [R, G, B]
pixel_y, pixel_x = 100, 200
print(f'Pixel [{pixel_y}, {pixel_x}]:', img_rgb[pixel_y, pixel_x])

Shape: (4032, 2268, 3)
Dtype: uint8
Pixel [100, 200]: [20 34 19]


## Etapa 3 — Separar canais R, G, B

In [ ]:
# Converte para float32 — necessário antes de dividir para não truncar decimais
img_f = img_rgb.astype(np.float32)

# Normaliza para [0, 1] dividindo por 255.0
# VARI foi projetado para refletância (0–1); sem isso o numerador pode chegar a 255,
# fazendo outliers explodirem até 255× mais quando o denominador é próximo de zero
img_f = img_f / 255.0

# Fatia o eixo 2: [:, :, n] = todas as linhas, todas as colunas, canal n
R = img_f[:, :, 0]  # vermelho
G = img_f[:, :, 1]  # verde
B = img_f[:, :, 2]  # azul

print('Shape de cada canal:', R.shape)
print(f'R — min: {R.min():.4f}, max: {R.max():.4f}')
print(f'G — min: {G.min():.4f}, max: {G.max():.4f}')
print(f'B — min: {B.min():.4f}, max: {B.max():.4f}')

## Etapa 4 — Calcular VARI

In [ ]:
# Numerador: pixels com G > R são positivos (vegetação); G < R são negativos (solo, estruturas)
# Denominador: com valores em [0,1] o epsilon 1e-6 tem peso proporcional real
vari_raw = (G - R) / (G + R - B + 1e-6)

print('--- Antes do clip ---')
print(f'VARI raw — min: {vari_raw.min():.2f}, max: {vari_raw.max():.2f}')
print(f'Pixels com vari_raw < -1: {(vari_raw < -1).sum()}')
print(f'Pixels com vari_raw >  1: {(vari_raw >  1).sum()}')

# Limita ao intervalo [-1, 1] — outliers de pixels saturados não distorcem o colormap
vari = np.clip(vari_raw, -1, 1)

print('\n--- Após o clip ---')
print(f'VARI — min: {vari.min():.6f}, max: {vari.max():.6f}, média: {vari.mean():.4f}')
print(f'Pixels com VARI > 0: {(vari > 0).sum()}')
print(f'Pixels com VARI < 0: {(vari < 0).sum()}')

## Etapa 4.1 — Máscara de luminância

In [ ]:
# Brilho médio por pixel — proxy de luminância com valores em [0, 1]
luminance = (R + G + B) / 3

# Pixels abaixo do threshold são sombras onde G+R-B≈0 não tem significado físico de vegetação
threshold = 0.1
mask_sombra = luminance < threshold

# Copia o VARI e zera as sombras — np.nan seria alternativa, mas 0 é mais seguro para colormap
vari_masked = vari.copy()
vari_masked[mask_sombra] = 0

total_pixels = mask_sombra.size
print(f'Pixels mascarados (sombra):   {mask_sombra.sum():>8} ({100 * mask_sombra.mean():.2f}%)')
print(f'Pixels válidos para análise:  {(~mask_sombra).sum():>8} ({100 * (~mask_sombra).mean():.2f}%)')
print()
print(f'VARI médio — com sombras:     {vari.mean():.4f}')
print(f'VARI médio — sem sombras:     {vari_masked[~mask_sombra].mean():.4f}')

## Etapa 5 — Calcular ExG

In [ ]:
# Com valores normalizados em [0,1], o range teórico do ExG passa a ser [-2, +2]
# Valores positivos = verde dominante (vegetação); negativos = vermelho/azul dominante
exg = 2 * G - R - B

print('ExG shape:', exg.shape)
print(f'ExG — min: {exg.min():.4f}, max: {exg.max():.4f}, média: {exg.mean():.4f}')
print(f'\nComparação de médias:')
print(f'  VARI (normalizado -1 a 1): {vari.mean():.4f}')
print(f'  ExG  (normalizado -2 a 2): {exg.mean():.4f}')

## Etapa 6 — Visualização comparativa

## Etapa 7 — Segmentação binária por ExG

In [ ]:
# ExG > 0 = excesso de verde sobre vermelho+azul → vegetação
# Zero é o corte natural da fórmula com valores normalizados em [-2, +2]
threshold_exg = 0.0
mascara_veg = exg > threshold_exg

# Converte para uint8 (0 ou 255) para visualização e salvamento via OpenCV
mascara_bin = mascara_veg.astype(np.uint8) * 255

pct_veg = mascara_veg.mean() * 100
print(f'Threshold ExG: {threshold_exg}')
print(f'Pixels classificados como vegetação: {mascara_veg.sum():>8} ({pct_veg:.2f}%)')
print(f'Pixels classificados como não-veg:   {(~mascara_veg).sum():>8} ({100 - pct_veg:.2f}%)')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img_rgb)
axes[0].set_title('Imagem original (RGB)', fontsize=13)
axes[0].axis('off')

axes[1].imshow(mascara_bin, cmap='gray')
axes[1].set_title(f'Segmentação ExG > {threshold_exg}  (branco = vegetação)', fontsize=13)
axes[1].axis('off')

# Sobrepõe a máscara em verde sobre a imagem original para validação visual
overlay = img_rgb.copy()
overlay[mascara_veg] = [0, 200, 80]
axes[2].imshow(overlay)
axes[2].set_title('Overlay — vegetação detectada (verde)', fontsize=13)
axes[2].axis('off')

plt.tight_layout()
plt.savefig('../outputs/segmentacao_exg.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/segmentacao_exg.png')

## Etapa 8 — Análise multi-imagem

In [ ]:
from pathlib import Path

def processar_imagem(path, threshold_exg=0.0, lum_threshold=0.1):
    img_bgr = cv2.imread(str(path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_f   = img_rgb.astype(np.float32) / 255.0

    R, G, B = img_f[:,:,0], img_f[:,:,1], img_f[:,:,2]

    # VARI com máscara de luminância
    vari_raw = (G - R) / (G + R - B + 1e-6)
    vari     = np.clip(vari_raw, -1, 1)
    mask_sombra = (R + G + B) / 3 < lum_threshold
    vari[mask_sombra] = 0

    # ExG e segmentação binária
    exg        = 2*G - R - B
    mascara_veg = exg > threshold_exg
    overlay    = img_rgb.copy()
    overlay[mascara_veg] = [0, 200, 80]

    return {
        'nome':       path.stem,
        'img_rgb':    img_rgb,
        'vari':       vari,
        'exg':        exg,
        'mascara':    mascara_veg,
        'overlay':    overlay,
        'pct_veg':    mascara_veg.mean() * 100,
        'vari_medio': vari[~mask_sombra].mean(),
    }

# Coleta imagens reais da pasta (exclui o arquivo de teste sintético)
imagens = sorted([p for p in Path('../images').iterdir()
                  if p.suffix in ('.jpeg', '.jpg', '.png') and p.stem != 'teste'])

resultados = [processar_imagem(p) for p in imagens]

# --- Tabela resumo ---
print(f'{"Imagem":<45} {"Veg %":>7} {"VARI médio":>12}')
print('-' * 66)
for r in resultados:
    print(f'{r["nome"]:<45} {r["pct_veg"]:>6.2f}% {r["vari_medio"]:>12.4f}')

# --- Grade visual: 1 linha por imagem, 3 colunas ---
fig, axes = plt.subplots(len(resultados), 3, figsize=(18, 6 * len(resultados)))

for i, r in enumerate(resultados):
    axes[i, 0].imshow(r['img_rgb'])
    axes[i, 0].set_title(f'{r["nome"]}\nOriginal', fontsize=11)
    axes[i, 0].axis('off')

    im = axes[i, 1].imshow(r['vari'], cmap='RdYlGn', vmin=-1, vmax=1)
    axes[i, 1].set_title(f'VARI  (médio: {r["vari_medio"]:.3f})', fontsize=11)
    axes[i, 1].axis('off')
    plt.colorbar(im, ax=axes[i, 1], fraction=0.046, pad=0.04)

    axes[i, 2].imshow(r['overlay'])
    axes[i, 2].set_title(f'ExG overlay  ({r["pct_veg"]:.1f}% vegetação)', fontsize=11)
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('../outputs/analise_multi_imagem.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/analise_multi_imagem.png')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Imagem original ---
axes[0].imshow(img_rgb)
axes[0].set_title('Imagem original (RGB)', fontsize=13)
axes[0].axis('off')

# --- Mapa VARI com máscara de luminância ---
im_vari = axes[1].imshow(vari_masked, cmap='RdYlGn', vmin=-1, vmax=1)
axes[1].set_title(f'VARI  (máscara luminância < {threshold})', fontsize=13)
axes[1].axis('off')
plt.colorbar(im_vari, ax=axes[1], fraction=0.046, pad=0.04)

# --- Mapa ExG ---
im_exg = axes[2].imshow(exg, cmap='RdYlGn')
axes[2].set_title('ExG  (Excess Green Index)', fontsize=13)
axes[2].axis('off')
plt.colorbar(im_exg, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('../outputs/comparativo_indices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/comparativo_indices.png')